## 一、什么是自然语言处理 NLP

自然语言处理（Natural Language Processing，NLP）是研究如何让计算机处理、理解和生成人类语言的技术领域。

| 概念   | 所在位置      | 主要回答的问题          |
| ---- | --------- | ---------------- |
| NLP  | 应用领域      | 要处理什么类型的问题       |
| 机器学习 | 技术方法      | 如何从数据中学习规律       |
| 深度学习 | 机器学习的一类方法 | 如何使用多层神经网络学习复杂规律 |

 **白话理解**：NLP 是“让计算机处理人话”这件事；机器学习和深度学习是完成这件事可以使用的方法。

## 三、把文本整理成可处理的数据

### 3.1 NLP 中的常见数据单位

| 名称   | 含义             | 示例            |
| ---- | -------------- | ------------- |
| 语料库  | 一批用于分析或训练的文本集合 | 一万条客服工单       |
| 文档   | 一条相对完整的文本      | 一条工单、一篇新闻     |
| 句子   | 文档中的一句话        | “订单一直没有发货。”   |
| 词或字符 | 更小的文本单位        | “订单”“发货”或单个汉字 |
| 标签   | 希望模型预测的答案      | “物流问题”        |

### 3.2 文本清洗

常见清洗操作包括：

- 去除没有业务意义的 HTML 标签和多余空白；
- 统一全角、半角和大小写；
- 根据业务规则处理网址、手机号和订单号；
- 修正常见乱码；
- 保留可能影响含义的标点和否定词

In [2]:
import re

from pandas._libs.tslibs import vectorized

In [3]:
def clean_text(text: str) -> str:
    """完成课堂示例所需的轻量清洗。"""
    # 将简单 HTML 标签替换为空格
    text = re.sub(r"<[^>]+>", " ", text)
    # 将连续空白合并成一个空格
    text = re.sub(r"\s+", " ", text)
    # 去除文本开头和结尾的空白
    return text.strip()

message = '  <p>订单  还没有发货！</p>  '
print(clean_text(message))

订单 还没有发货！


有些项目会删除“的”“了”“是”等高频但区分作用较弱的停用词。停用词表必须结合业务确定，不能直接套用通用名单。对于分类任务，TF-IDF 本身也会降低大量常见词的权重，因此课堂基线暂不额外删除停用词。

### 3.4 使用 jieba 观察分词结果

In [4]:
import jieba

message = '商品已经退回但退货退款还没到账'

# lcut() 直接返回分词后的列表
print(jieba.lcut(message))

# 将业务专有表达加入词典
jieba.add_word('退货退款')
print(jieba.lcut(message))

Building prefix dict from the default dictionary ...
Dumping model to file cache C:\Users\ADMINI~1\AppData\Local\Temp\jieba.cache
Loading model cost 0.813 seconds.
Prefix dict has been built successfully.


['商品', '已经', '退回', '但', '退货', '退款', '还', '没到', '账']
['商品', '已经', '退回', '但', '退货退款', '还', '没到', '账']


## 四、传统方法如何表示文本

### 4.1 词袋模型

假设词表为：

```plain
["订单", "退款", "物流", "延迟"]
```

“订单物流延迟”可以表示为：

```plain
分词结果：订单 / 物流 / 延迟

对应词表：
[1, 0, 1, 1]
```

### 4.2 TF-IDF

TF-IDF 用来衡量一个词对某篇文档的重要程度：

- 一个词在当前文档中频繁出现，重要程度可能更高；
- 一个词在所有文档中都很常见，区分能力会降低。

In [6]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

texts = [
    "订单一直没有发货",
    "申请退款不要这个商品了",
    "账号登录失败",
]

# analyzer="char"：以中文字符为基本单位，不需要额外分词
# ngram_range=(1, 2)：同时使用单个字符和相邻两个字符
vectorizer = TfidfVectorizer(
    analyzer='char',
    ngram_range=(1, 2),
)

# fit：根据文本建立特征词表
# transform：将文本转换成 TF-IDF 特征矩阵
text_features = vectorizer.fit_transform(texts)
print('特征矩阵形状: ', text_features.shape)
print('部分特征: ', vectorizer.get_feature_names_out()[:10])

# 将小规模示例转换成 DataFrame，直观看到部分 TF-IDF 权重
feature_names = vectorizer.get_feature_names_out()
feature_table = pd.DataFrame(
    text_features.toarray(),
    index=texts,
    columns=feature_names,
)

# 只选择几个容易理解的字符或字符组合进行展示
selected_features = [
    feature
    for feature in ["订", "订单", "发", "发货", "退", "退款", "账", "账号", "登录"]
    if feature in feature_table.columns
]
print(feature_table[selected_features].round(3))

特征矩阵形状:  (3, 47)
部分特征:  ['一' '一直' '不' '不要' '个' '个商' '了' '单' '单一' '发']
                 订     订单      发     发货      退     退款      账     账号     登录
订单一直没有发货     0.258  0.258  0.258  0.258  0.000  0.000  0.000  0.000  0.000
申请退款不要这个商品了  0.000  0.000  0.000  0.000  0.218  0.218  0.000  0.000  0.000
账号登录失败       0.000  0.000  0.000  0.000  0.000  0.000  0.302  0.302  0.302


text_features 是稀疏矩阵，只保存非零值，适合存储大量文本特征。这里使用字符及相邻字符组合建立课堂基线，不依赖额外的中文分词库。

TfidfVectorizer 的词表只能从训练集学习。验证集或新文本中完全没有见过的特征不会自动加入词表，这类词通常称为未登录词。字符级特征能够利用其中已经见过的字符或字符组合，但仍不能真正理解词序、上下文和语义。

## 五、NLP 项目的基本流程

需要特别注意：

- 同一用户或重复工单不能随意同时出现在训练集和测试集中；
- `TfidfVectorizer` 只能在训练集上执行 `fit`；
- 使用 `Pipeline` 可以把向量化和模型训练连接起来，减少数据泄漏；
- 验证集用于比较方案和分析错误，测试集只在方案确定后使用一次；
- 不能只看 Accuracy，还要检查各类别的 Precision、Recall 和 F1；
- 文本中的订单号、手机号等信息要脱敏；
- 线上出现的新表达要持续记录并加入评测集。
